<a href="https://colab.research.google.com/github/Albrianp/data-science-2026/blob/main/Pertemuan3_Albrian_Pikikene_240401010160.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pertemuan ke-3

*   Nama : Albrian Pikikene
*   NIM : 240401010160
*   Kelas : IF405

In [1]:
# --- Generate dataset housing_dirty.csv (jalankan sekali di awal) ---
import pandas as pd, numpy as np

np.random.seed(42)
n = 130

kota_list = ['jakarta', ' Bandung', 'surabaya ', 'Yogyakarta', ' medan ']
kondisi_list = [' Baik', 'sedang ', ' RUSAK', 'baik', 'Sedang']

df = pd.DataFrame({
    'id': range(1, n+1),
    'luas_m2': np.random.normal(150, 60, n).round(1),
    'harga_juta': np.random.normal(700, 300, n).round(1),
    'kota': np.random.choice(kota_list, n),
    'kamar': np.random.choice([1,2,3,4,5,6], n),
    'tahun_bangun': np.random.randint(1990, 2023, n),
    'kondisi': np.random.choice(kondisi_list, n),
})

# suntikkan missing values
for col, jumlah in [('luas_m2', 18), ('harga_juta', 17), ('kamar', 10)]:
    idx = np.random.choice(df.index, jumlah, replace=False)
    df.loc[idx, col] = np.nan

# suntikkan outlier ekstrem
df.loc[0, 'luas_m2'] = -50
df.loc[1, 'luas_m2'] = 9500
df.loc[2, 'harga_juta'] = -500
df.loc[3, 'harga_juta'] = 100_000_000
df.loc[4, 'tahun_bangun'] = 9999

# suntikkan beberapa duplikat
df = pd.concat([df, df.iloc[[5, 6]]], ignore_index=True)

df.to_csv('housing_dirty.csv', index=False)
print('housing_dirty.csv berhasil dibuat, shape:', df.shape)

housing_dirty.csv berhasil dibuat, shape: (132, 7)


In [2]:
import pandas as pd, numpy as np
from scipy.stats.mstats import winsorize

In [3]:
df = pd.read_csv('housing_dirty.csv')
print('Shape awal:', df.shape)
print(df.describe())
print(df.isnull().sum())

Shape awal: (132, 7)
               id      luas_m2    harga_juta       kamar  tahun_bangun
count  132.000000   114.000000  1.140000e+02  122.000000    132.000000
mean    64.606061   227.057018  8.778927e+05    3.606557   2066.492424
std     38.076812   878.031699  9.365792e+06    1.708304    695.774690
min      1.000000   -50.000000 -5.000000e+02    1.000000   1990.000000
25%     31.750000   114.825000  4.696250e+02    2.000000   1997.750000
50%     64.500000   146.800000  7.537000e+02    4.000000   2006.000000
75%     97.250000   178.725000  9.021750e+02    5.000000   2015.000000
max    130.000000  9500.000000  1.000000e+08    6.000000   9999.000000
id               0
luas_m2         18
harga_juta      18
kota             0
kamar           10
tahun_bangun     0
kondisi          0
dtype: int64


In [14]:
# STEP 1 — Hapus Duplikat
df.drop_duplicates(inplace=True)
print('Setelah hapus duplikat:', df.shape)

Setelah hapus duplikat: (130, 7)


In [13]:
# STEP 2 — Normalisasi String
df['kota'] = df['kota'].str.strip().str.title()
df['kondisi'] = df['kondisi'].str.strip().str.lower()

In [12]:
# STEP 3 — Imputasi Missing Values
df['luas_m2'] = df['luas_m2'].fillna(df['luas_m2'].median())
df['harga_juta'] = df['harga_juta'].fillna(df['harga_juta'].median())
df['kamar'] = df['kamar'].fillna(df['kamar'].mode()[0])

In [11]:
# STEP 4 — Tangani Outlier (IQR Fence)
for col in ['harga_juta', 'luas_m2', 'tahun_bangun']:
  Q1, Q3 = df[col].quantile([0.25, 0.75])
  IQR = Q3 - Q1
  df[col] = df[col].clip(Q1-1.5*IQR, Q3+1.5*IQR)

In [10]:
# STEP 5 — Validasi & Ekspor
assert df.isnull().sum().sum() == 0, 'Masih ada missing!'
assert df.duplicated().sum() == 0, 'Masih ada duplikat!'
print('Shape akhir:', df.shape)
df.to_csv('housing_clean.csv', index=False)
print('Dataset bersih tersimpan!')

Shape akhir: (130, 7)
Dataset bersih tersimpan!


In [9]:
import requests, pandas as pd
from pandas import json_normalize

URL = "https://jsonplaceholder.typicode.com/users"
response = requests.get(URL, timeout=10)

if response.status_code == 200:
    data = response.json()
    df_api = json_normalize(data)
    print(df_api.head())
else:
    print("Error:", response.status_code)

   id              name   username                      email  \
0   1     Leanne Graham       Bret          Sincere@april.biz   
1   2      Ervin Howell  Antonette          Shanna@melissa.tv   
2   3  Clementine Bauch   Samantha         Nathan@yesenia.net   
3   4  Patricia Lebsack   Karianne  Julianne.OConner@kory.org   
4   5  Chelsey Dietrich     Kamren   Lucio_Hettinger@annie.ca   

                   phone        website     address.street address.suite  \
0  1-770-736-8031 x56442  hildegard.org        Kulas Light      Apt. 556   
1    010-692-6593 x09125  anastasia.net      Victor Plains     Suite 879   
2         1-463-123-4447    ramiro.info  Douglas Extension     Suite 847   
3      493-170-9623 x156       kale.biz        Hoeger Mall      Apt. 692   
4          (254)954-1289   demarco.info       Skiles Walks     Suite 351   

    address.city address.zipcode address.geo.lat address.geo.lng  \
0    Gwenborough      92998-3874        -37.3159         81.1496   
1    Wisokyburgh